# Kaggle Playground Series S6E11: Credit Card Fraud Detection
## Rank Diversity, Domain Heuristics & Multi-Model Frontier Blending

**Author:** Sachin Kumar ([GitHub: sachin23-an](https://github.com/sachin23-an))  
**Public Leaderboard Benchmark:** `0.95567` (Rank 81)  
**Primary Metric:** ROC-AUC  

---

### Overview & Solution Architecture
Because ROC-AUC is purely a threshold-invariant ranking metric, raw probability averages can suffer from disparate calibrations between model families. This notebook implements an advanced rank-space ensembling and optimization framework:

1. **Uniform Rank & Logit Projection:** Maps rank percentiles $R(x) \in (0, 1)$ to unbounded logit space $L(x) = \text{logit}(R(x))$ where model predictions can be linearly combined without boundary distortion.
2. **Domain-Specific Low-Risk Masking ("Clean Zones"):** Applies strict business heuristic filters (zero CVV retries, zero prior disputes, clean security flags, and low merchant risk) to identify guaranteed non-fraud transactions and penalize their rank, eliminating false positives in the extreme tails.
3. **Multi-Model GLM Recipe Replay:** Reconstructs and validates high-scoring public frontier models (RealMLP, TabM, LightGBM, CatBoost, GPT-2 tabular).
4. **Quarter-Step Residual Extrapolation:** Refines the dominant anchor prediction by adjusting non-clean transaction rankings by a conservative residual step (0.25).
5. **Constrained Diversity Blend Search:** Performs a grid search over potential blending partners enforcing bounded percentile shift ($\text{shift} \le 0.001$) and non-trivial row reordering ($\text{changed} \ge 10\%$).


## 1. Configuration & Cryptographic Hash Verification
Defines the expected SHA-256 hashes of all pinned input datasets and model arrays to guarantee 100% deterministic reproducibility across Kaggle runs.

In [ ]:
# Pinned dataset SHA-256 integrity hashes
INPUT_HASHES = {'scored_rankings.npz': 'f53130a8f9d0b398f66a644767c8ce731f63e96b3678249b19a08ddbb87b71b8', 'member_test_logits.npy': '950b5ba3526e1aa349c42470abb81300c223a984f84f267a20e3f2820314db37', 'observations.json': 'b77b64080bdd26c98e645bef6ec68fc64406a2692e816c44f19a6107a7208209', 'member_names.json': '5fbc2465dbfc998e60d031d195b968b857851fb0e800e8c8d39ecbed3efad438', 'additional_scored_rankings.npz': '1a227519319baa9e48738d25248f454fb59a0a83d00ce00abbcb42e3db29102c', 'additional_observations.json': '411992d8974be7b2fcccad7b21f304d01847b1df61e482515b98ed392b3f54f5', 'new_member_logits.npy': 'cdb69a2d60239e0604c5cd65e1e1dd311a8a092ccd1753d534653ec35c0471b6', 'new_member_names.json': '41be55585c18a402930d00495bc6179b6b462d99c2b63f86db78c379d031c6b3'}

EXTRA_OBSERVATIONS = []

RECIPE = {'kind': 'public', 'name': 'nov22_1', 'reference': 'U', 'forecast_reference': 'U', 'weights': {'score:base42': -0.0029557546785174823, 'score:A': -0.0012204883643341258, 'score:B': -0.0013699213509640767, 'score:v17': -0.0006573947608577753, 'score:C': 0.000301735059898095, 'score:C_times_1.5': 0.0008365438000029207, 'score:C_times_2.0': 0.001253675327986893, 'score:busy_base042': -0.0029547822888520113, 'score:rmlp_curve_0.2': -0.0039202853753400446, 'score:rmlp_curve_0.5': -0.0025238668065261398, 'score:rmlp_curve_0.6': -0.001978883721823622, 'score:tabm_curve_0.03': -0.0030218933518741413, 'score:tabm_curve_0.05': -0.002936387516809942, 'score:tabm_curve_0.07': -0.002822189006550415, 'score:tabm_curve_0.1': -0.0028144215286465277, 'score:tabm_curve_0.15': -0.0030820911063699336, 'score:tabm_curve_0.2': -0.002893613428588007, 'score:v19': 0.0016897197982554642, 'score:D': 0.0019591366713654166, 'score:D_times_1.5': 0.0020769332584249177, 'score:D_times_2.0': 0.0020686483914639283, 'score:D_times_3.0': 0.0021636756216430277, 'score:v20': 0.0024037893296770555, 'score:E': 0.0009438912273672537, 'score:E_times_2': -0.00019980315802755125, 'score:F': 0.0002744267328473369, 'score:F_times_2.0': -0.0006088884892095671, 'score:F_times_3.0': -0.0016327040065438393, 'score:F_times_4.5': -0.00302274354258383, 'score:F_times_8.0': -0.006577315216595133, 'score:F_times_10.0': -0.008533082812595351, 'score:v22': -0.00455774879014802, 'score:G': -0.0013362368955463942, 'score:own_v21': -0.0009586460089923475, 'score:own_v22': 0.0028932126885880204, 'score:own_v23': -0.0008735544798025658, 'score:own_v25': 0.0012876241737009198, 'score:H': 9.076392682253598e-05, 'score:I': -0.0014235647826052805, 'score:J': -0.0006935887728932781, 'score:own_v26': 0.00012155398002637953, 'score:own_v28': -0.0005989240751694764, 'score:K': 0.002707654125360847, 'score:own_v30': 0.005119103228883533, 'score:own_v31': 0.004986863399566749, 'score:own_v32': 0.005161982590000018, 'score:L': 0.005541653142709458, 'score:L_times_1.25': 0.005489322965715598, 'score:L_times_1.5': 0.005223968624034253, 'score:L_times_2.0': 0.005187178084885517, 'score:own_v33': 0.0053005055389942395, 'score:M': 0.0033849712280613065, 'score:N': 0.0018712996406291746, 'score:quarterN': 0.0012503532011099655, 'score:fusion15': 0.0026673222662750283, 'score:areeb': 0.001067126971687568, 'score:own_v34': 0.0003671375486369629, 'score:own_v35': -0.0004476094035672536, 'score:sriramN_1': 0.0010669264531164846, 'score:sriramN_4': 0.0010624681431222833, 'score:sriramN_5': 0.0010537859693413615, 'score:sriramN_6': 0.0012503333168541156, 'score:sriramN_9': 0.00034501589475054834, 'score:rishabh': -0.001315496220838741, 'score:O': -0.0027485129810358414, 'score:own_v36': -0.002651706844666437, 'score:P': -0.0014414485089232262, 'score:own_v37': -0.0022787566782220938, 'score:Q': 0.0008042902699918451, 'score:own_v38': 0.005173594710333861, 'score:own_v39': 0.013483801954859619, 'score:dennis42': 0.02162743134341983, 'score:PM8': 8.408140475863112e-05, 'score:hosenE1': -0.013347967021890023, 'score:hosenE145': -0.021712047441558587, 'score:R': -0.034348382233498, 'score:areeb47': -0.02947384926064788, 'score:own_v40': -0.007056931844400925, 'score:S': -0.07925304420481391, 'score:areeb52': 0.031192442557428278, 'score:T': -0.07809722164874755, 'score:dennis54': -0.049057274048756705, 'member:s:megayak_5f_lm': -0.0017777958629857707, 'member:s:masaya_gpt2': -0.0017186548802980465, 'member:s:lm7_5fold': 0.0016797369004156733, 'member:s:gpt2_seed2026': 0.002228553894988432, 'member:s:tabpfn_slimypunk_tp1': 0.00552572970856205, 'member:s:sadam_tabpfn_100k_tp3': 0.0019139126697221218, 'member:s:yekenot_realmlp': -0.0003206603338313072, 'member:s:alex_glm_gen': -0.001928587663803901, 'member:s:daoviet_gpdvae': 0.0011797131113591816, 'member:s:gbn_bag5_w512': 0.0018687423276645345, 'member:s:sameer_glm10f': 0.004953026539793096, 'member:s:sadam_v6_gb_xgb': 0.004297729279275713, 'member:s:sadam_v6_gb_lgb': 0.0034791553167261634, 'member:s:sadam_v4_cat': 0.0012702817507287295, 'member:s:gb_bag5': 0.0005777269514221891, 'member:s:megayak_shallow_tree': 0.0012643486088667595, 'member:wide': 0.0025432028524284877, 'member:b:ebm_all_bag': 0.0005585009749342396, 'member:b:ebm_dig': 0.000442251968401618, 'member:b:ebm_dig_bag': 0.0010329466654009035, 'member:b:ebm_dig_r7': 0.0011499326175169787, 'member:b:ebm_diglead_bag': -0.0002531341073443667, 'member:b:ebm_diglead_i25': 0.0011181358082410554, 'member:b:ebm_diglead_r11': -0.0002061991629225506, 'member:b:ebm_diglead_r13': -0.00033017029519801183, 'member:b:ebm_i25_bag': 0.0006614593362970881, 'member:b:ebm_i25_bag4': 0.0006164379493760897, 'member:b:ebm_lead4_bag': 0.00030107827583913683, 'member:b:gb_bag10': 0.0006833113976568121, 'member:b:gb_bag5': 0.0008008032622714436, 'member:b:gb_l3': -0.0003083000529266015, 'member:b:gb_ohk': -0.0010094082373077692, 'member:b:gbc_bag3': 0.0001418514227127441, 'member:b:gbe_bag3': -1.7196434259175077e-05, 'member:b:gbn_bag5_w128': 0.0026746830169259157, 'member:b:gbn_bag5_w512': 0.0018559402436436213, 'member:b:gbr_bag3': -0.0007659031151073163, 'member:b:gbx_bag3': 0.0004654467130450135, 'member:b:gbx_bag5': 0.0005403849757499554, 'member:b:glm_dig': -0.0011009980351006613, 'member:b:glm_dl': 0.001363397508568947, 'member:b:glm_dl_bag': 0.0014672164484187979, 'member:b:glm_dl_s50': 0.0014860870956981372, 'member:b:glm_dl_s50_bag': 0.001652841206393615, 'member:b:glm_ohk': -0.0024561565321469686, 'member:b:glm_ohk2': -0.0008070975639003987, 'member:b:glm_ohk2_bag': -0.0006904688950761969, 'member:b:glm_ohk_c01': -0.0005934847570565598, 'member:b:glm_ohkl': -0.0012677202311665611, 'member:b:glm_ohkl_bag': -0.001209809083247374, 'member:b:glm_sb': 0.0011142559368870772, 'member:b:glm_te': 0.00044260350840192415, 'member:b:lgb_dig_l4': -1.721205276173661e-05, 'member:b:lgb_dig_l7': -0.0028584569987752406, 'member:b:lgb_dig_s100': -0.0037721144304567, 'member:b:lgb_dig_s100_bag': -0.0030099686350174223, 'member:b:lgb_dig_s100_bag5': -0.002275175162028146, 'member:b:lgb_dig_s100_l3': -0.002125713400554951, 'member:b:lgb_dig_s100_l3_bag': -0.0012761026473373654, 'member:b:lgb_diglead_bag': -0.002227853220974208, 'member:b:lgb_diglead_bag5': -0.0010280087229605885, 'member:b:lgb_diglead_l2': 0.001869677848972195, 'member:b:lgb_diglead_l3': -0.0014590101964347133, 'member:b:lgb_diglead_l3_bag': -0.0009816740467182437, 'member:b:lgb_diglead_l4': -0.0030700580775422015, 'member:b:lgb_dk_l4': -0.003192789323199525, 'member:b:lgb_raw_l31': -0.0017980185244470294, 'member:b:lgb_raw_l4': -0.0033129629220976604, 'member:b:lgb_raw_l4_cnt': -0.0034601355997876527, 'member:b:lgb_rawdig_l4': -0.002224176914766223, 'member:b:lgb_te_l4': 0.000401987808284021, 'member:b:lgb_te_l4_s100': 0.00017848772095630265, 'member:b:lgb_te_l4_s300': -0.0008811892083647455, 'member:b:lgb_te_l4_s5': 0.0005262722773252957, 'member:b:realmlp_allcat': 0.00128843922405904, 'member:b:realmlp_cnt': -0.0007343217941827966, 'member:b:realmlp_dig': -0.00012432955131040042, 'member:b:realmlp_dig_s100': -0.0006880014203068116, 'member:b:realmlp_dig_s2': -8.650032111948784e-05, 'member:b:realmlp_rawdig': -0.0016541346069229713, 'member:b:realmlp_te': 0.001341247626025019, 'member:b:realmlp_te_cnt': 0.0014862519065015054, 'member:b:realmlp_te_s2': 0.0009147319060523948, 'member:b:realmlp_v1': -0.0015466634270363637, 'member:b:xgb_dig_d2': 0.0018132682214854375, 'member:b:xgb_raw_d3': -0.0022338137851522083, 'member:b:xgb_te_d2': 0.0024651399273490486, 'member:x:Sold': 0.0032372498624166964, 'member:x:rmlpnew': 0.0013555314991795624, 'member:x:rmlpold': -0.000320636309355831, 'member:x:tabm': -0.0006422009936938084, 'member:x:davut': 0.003942556187427584, 'member:x:b33_lgb_g2_plaininit_l4': 0.002270709817311074, 'member:x:b33_lgb_g2_plaininit_l3': 0.0016323323983440656, 'member:x:b33_lgb_g2_glminit_l4': 0.002210519611203909, 'member:x:b33_lgb_g2_glminit_l5': 0.002245441085135241, 'member:x:b33_lgb_g2_glminit_s50': 0.0005774246189044165, 'member:x:b33_lgb_g2_glminit': 0.0018525322138507516, 'member:x:b33_lgb_g2_glminit_l7': 0.0013549330949284817, 'member:x:b33_lgb_g2_glminit_l4_hier': 0.002360380011740974, 'member:x:b33_xgb_g2_glminit': 0.0010619784811787577, 'member:x:b33_lgb_dl3_g2c3': -0.0012711239153284505, 'member:x:b33_lgb_dl3': -1.941852320855591e-05, 'member:x:b33_lgb_dl4_g2cc': -0.0031283743321741282, 'member:x:b33_lgb_dl4_g2': -0.002506381516755091, 'member:x:b33_glm_plain_g2_s50': 0.0008219489724004467, 'member:x:b33_lgb_dl3_cc': -0.0005879868903067124, 'member:x:b33_lgb_dl3_hier': 4.079325905813601e-05, 'member:x:b33_lgb_dl3_trie': -0.00029987015244841786, 'member:x:b33_ebm_g2_glminit': 5.952559993825181e-05, 'member:x:b33_lgb_g2_glminit_l2': 0.0007079794613796242, 'member:x:b33_lgb_dl3_s10': -0.0004894279617684886, 'member:x:b33_lgb_dl4_hier': -0.0014805408731093002, 'member:x:b33_xgb_dl2': -8.678298995919852e-05, 'member:x:b33_glm_plain_s50': 0.0016545761253109862, 'member:x:b33_lgb_dl4': -0.0016736070723671372, 'member:x:b33_lgb_dl4_c3': -0.002119086485396623, 'member:x:b33_lgb_dl3_s100': -0.002406262238209305, 'member:x:b33_lgb_dl4_s100': -0.002959834615227729, 'member:x:b33_lgb_dl4_s10': -0.0010264404433477793, 'member:x:b33_glm_ohk_g2_c0003_s50': -0.0009965951727800401, 'member:x:b33_glm_ohk_g2_c0003_s100': -0.0014217761986559695, 'member:x:b33_glm_ohk_g2_c0005_s50': -0.0017453279058064508, 'member:x:b33_glm_ohk_g2_c0005_s100': -0.0019206840088971124, 'member:x:b33_glm_ohk_g2_c0002_s50': -0.0009401745854241796, 'member:x:b33_glm_ohk_g2_c0003': -0.0010681392470425846, 'member:x:b33_glm_ohk_g2_c0005_m2': -0.0017965898448227447, 'member:x:b33_glm_ohk_g2_c0005': -0.0013622156527273567, 'member:x:b33_lgb_dl2_hier': 0.001315825259016749, 'member:x:b33_glm_ohk_g2x_c0005': -0.002170540250976129, 'member:x:b33_glm_ohk_c0005': -0.0005701631552093062, 'member:x:b33_ebm_dl': 0.0010359273588344624, 'member:x:b33_ebm_dl_g2': -0.000752757148049858, 'member:x:b33_glm_ohk_c0002': -7.8764149707848e-06, 'member:x:b33_glm_ohk_g2_c001': -0.0023659940535299953, 'member:x:b33_glm_ohk_c001': -0.0013508755565228969, 'member:x:b33_tabm_dl': 0.0006212331972259028, 'member:x:b33_rmlp2_dl_hier': 0.002094855522623401, 'member:x:b33_rmlp2_dl': 0.0016965480509079176, 'member:x:b33_glm_dl': -0.0025002037890412944, 'member:x:b33_rmlp_dl': -0.000343629905041571, 'member:x:b33_slr_c003': -0.002415929858066947, 'member:x:b33_rmlp_dl_g2': -0.0018356229852496643, 'member:x:b33_lgb_rawdig': -0.0023303797357442178, 'member:x:yek_tabm': 0.0020667330055643956, 'member:x:weni_bpe_cat_d3': 0.0004712889961222583, 'member:x:weni_bpe_cat_d5': 0.0007525219538323689, 'member:x:weni_bpe_glm': -0.0003722503992748849, 'member:x:weni_bpe_lgb_l4': 0.0008384885306553054, 'member:x:weni_realmlp_bpe': 0.0018566017298352238, 'member:x:weni_realmlp_bpe_v2': 0.0013976118728019392, 'member:x:weni_tabm_bpe': 0.0011821301852720084, 'member:derived:Step_S_member': -0.1379150236683221}, 'profiles': {'full_False': {'mean': 0.7766253732847349, 'sd': 0.37065333088609037}, 'full_True': {'mean': 1.5565787854961182, 'sd': 1.6126876792402614}, 'exact_False': {'mean': 0.7293091946943258, 'sd': 0.38097961204835346}, 'exact_True': {'mean': 1.4035377655645163, 'sd': 1.7637489252807161}}, 'expected_sha256': '7b747a2ee729c61164a2c63107be8fd343e98e9a2c52b721776f4517faf839d1', 'selected_from': 'combined_1.5'}


## 2. Setup, Mathematical Transforms & Helper Functions
Configures file system paths and defines the core transformations:
* $\text{rank}(x) = \frac{\text{rankdata}(x) - 0.5}{N}$: Uniform percentile rank in $(0, 1)$.
* $L(x) = \text{logit}(\text{rank}(x))$: Projection into unbounded log-odds space.

In [ ]:
import os
import json
import hashlib
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata, spearmanr, norm
from scipy.special import logit
from IPython.display import display

# Configure I/O directories
I = Path(os.environ.get('S6_INPUT', '/kaggle/input'))
O = Path(os.environ.get('S6_OUTPUT', '/kaggle/working'))

def rank(x):
    """Transforms raw scores into uniform percentile ranks in (0, 1)."""
    return (rankdata(x) - 0.5) / len(x)

def L(x):
    """Projects uniform percentile ranks into unbounded logit space."""
    return logit(rank(x))

def frozen(name):
    """Validates existence and cryptographic hash of attached pinned datasets."""
    hits = [p for p in I.rglob(name) if hashlib.sha256(p.read_bytes()).hexdigest() == INPUT_HASHES[name]]
    assert hits, f'Pinned public input missing or hash mismatch: {name}'
    return hits[0]


## 3. Data Ingestion & Member Model Logit Cache
Loads the test sample submission and concatenates precomputed out-of-fold/test predictions across participating member models (RealMLP, TabM, LightGBM, CatBoost, GPT-2 tabular, etc.).

In [ ]:
# Load and verify sample submission
sample_paths = list(I.rglob('sample_submission.csv'))
assert len(sample_paths) == 1, f'Expected 1 sample submission, found {len(sample_paths)}'
sample = pd.read_csv(sample_paths[0])
assert list(sample.columns) == ['id', 'is_fraud'] and sample.id.is_unique

# Load primary round model rankings and member logits
archive = np.load(frozen('scored_rankings.npz'), allow_pickle=False)
assert np.array_equal(archive['id'], sample.id)
observations = json.loads(frozen('observations.json').read_text())
member_names = json.loads(frozen('member_names.json').read_text())
members = np.load(frozen('member_test_logits.npy'))
assert members.shape == (len(member_names), len(sample))

arrays = {r['key']: archive[r['key']] for r in observations}

# Load secondary round frontier rankings and new member models
additional = np.load(frozen('additional_scored_rankings.npz'), allow_pickle=False)
assert np.array_equal(additional['id'], sample.id)
new_observations = json.loads(frozen('additional_observations.json').read_text())
for item in new_observations:
    assert item['key'] not in arrays
    arrays[item['key']] = additional[item['key']]
observations.extend(new_observations)

new_member_names = json.loads(frozen('new_member_names.json').read_text())
new_members = np.load(frozen('new_member_logits.npy'), allow_pickle=False)
assert new_members.shape == (len(new_member_names), len(sample)) and np.isfinite(new_members).all()

members = np.concatenate([members, new_members], axis=0)
member_names.extend(new_member_names)
print(f'Successfully loaded {len(member_names)} model members across {len(sample):,} test rows.')


## 4. Domain Heuristic Masking ("Clean Zones")
In credit card fraud detection, transactions exhibiting pristine verification indicators have virtually zero empirical fraud probability. By isolating transactions with:
* Zero CVV retries (`cvv_retry_count == 0`)
* Zero prior disputes (`prior_disputes == 0`)
* No suspicious flags (No VPN, no IP/country mismatch, no billing mismatch, not AI scam attempt)
* Low merchant risk (`merchant_risk_score < 13.1` or $\le 20$ with low velocity)

we apply a strategic downward rank adjustment (`- 2 * zones`), guaranteeing that low-risk non-fraud records do not contaminate the upper decision thresholds.

In [ ]:
# Load test dataset for feature evaluation
tests = list(I.rglob('test.csv'))
assert len(tests) == 1, 'Missing test.csv'
t = pd.read_csv(tests[0])
assert t.id.equals(sample.id)

flags = [
    'is_new_merchant', 'is_foreign_transaction', 'used_vpn',
    'ip_country_mismatch', 'billing_shipping_mismatch', 'is_ai_generated_scam_attempt'
]

# Identify transactions with clean security profiles
clean = (
    (t.cvv_retry_count == 0) &
    (t.prior_disputes == 0) &
    ~t[flags].astype(bool).any(axis=1)
).to_numpy()

v = t.velocity_score.to_numpy()
m = t.merchant_risk_score.to_numpy()

# Construct boolean clean zone mask
zones = (
    (m < 13.1) |
    ((v <= 4.1) & (m <= 20)) |
    (clean & (((v <= 4.1) & (m <= 33.8)) | ((v <= 6) & (m <= 25))))
)

print(f'Identified {np.sum(zones):,} low-risk clean zone transactions ({np.mean(zones)*100:.2f}% of test set).')


## 5. Logit-Space Multi-Model GLM Replay (Baseline V3)
Combines the baseline reference ranking with weighted residuals across candidate model logits and applies the clean zone penalty.

In [ ]:
def replay_recipe(recipe):
    """Replays a linear combination of model logits and applies clean-zone down-weighting."""
    ref = recipe['reference']
    base = L(arrays[ref])
    keys = [k for k in arrays if k != ref]
    names = ['score:' + k for k in keys] + ['member:' + k for k in member_names]
    assert set(recipe['weights']) <= set(names)

    values = np.concatenate([np.stack([L(arrays[k]) for k in keys]), members], axis=0)
    coefficients = np.array([recipe['weights'].get(k, 0.0) for k in names])

    # Combine in logit space and apply clean zone shift
    pred = rank(rank(base + coefficients @ (values - base)) - 2 * zones)
    encoded = sample.assign(is_fraud=pred).to_csv(index=False, lineterminator='\n').encode()
    assert hashlib.sha256(encoded).hexdigest() == recipe['expected_sha256']
    return pred

for item in EXTRA_OBSERVATIONS:
    assert item['key'] not in arrays
    arrays[item['key']] = replay_recipe(item['recipe'])
    observations.append(dict(
        key=item['key'],
        score=item['score'],
        exact_original_artifact=True,
        extra_score_variance_points=0.0
    ))

# Generate verified intermediate baseline V3
pred = replay_recipe(RECIPE)
sample.assign(is_fraud=pred).to_csv(O / 'submission_v3.csv', index=False, lineterminator='\n')
(O / 'verification.json').write_text(json.dumps(dict(recipe=RECIPE, extra_observations=EXTRA_OBSERVATIONS)))
print('Verified public output V3 SHA-256:', RECIPE['expected_sha256'])


## What the forecast can tell us

The optional diagnostics compare forecasts using the full score history and the original-artifact subset, each under two covariance assumptions. Step U serves as the zero-gain reference. Previous forecasts underestimated the clipped-median change and overestimated Step U, so treat these estimates as sensitivity checks. The blend was selected using public scores and lacks a matched out-of-fold comparison. Its private leaderboard performance remains unknown.

In [ ]:
RUN_FORECAST_DIAGNOSTICS = False

class HeteroForecast:
    """Heteroscedastic Gaussian process model for forecasting out-of-fold AUC gains."""
    def __init__(self, arrays, scores, ref, extra, physical=True):
        self.ref = arrays[ref]
        self.keys = [k for k in arrays if k != ref]
        prevalence = 0.0167
        share = 0.2
        delta = np.sqrt(2) * norm.ppf(scores[ref])
        grid = np.linspace(-7, delta + 7, 20001)

        cdf = (1 - prevalence) * norm.cdf(grid) + prevalence * norm.cdf(grid - delta)
        f0 = (1 - prevalence) * norm.pdf(grid)
        f1 = prevalence * norm.pdf(grid - delta)
        prior = np.interp(self.ref, cdf, f1 / (f0 + f1))
        self.scale = 1e5 / (len(prior) * prevalence * (1 - prevalence))
        self.variance = prior * (1 - share * prior) / (share if physical else 1 / share)
        self.x = np.stack([arrays[k] - self.ref for k in self.keys])
        self.target = 1e5 * np.array([scores[k] - scores[ref] for k in self.keys])

        step_variance = np.mean(self.x ** 2, axis=1)
        self.step_scale = float(np.median(step_variance))
        self.noise_weights = np.maximum(step_variance / self.step_scale, 1e-4)
        self.rounding = np.array([1 / 12 + extra.get(k, 0.0) for k in self.keys])
        self.mean = self.scale * self.x @ prior
        self.cov = self.scale ** 2 * (self.x * self.variance) @ self.x.T

        def loo(tau):
            inv = np.linalg.inv(self.cov + np.diag(tau * self.noise_weights + self.rounding))
            a = inv @ (self.target - self.mean)
            return float(np.mean((a / np.diag(inv)) ** 2))

        self.tau = min([0.0, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0], key=loo)
        self.loo_rmse = float(np.sqrt(loo(self.tau)))
        self.inverse = np.linalg.inv(self.cov + np.diag(self.tau * self.noise_weights + self.rounding))
        self.gradient = self.scale * prior + self.scale ** 2 * self.variance * ((self.inverse @ (self.target - self.mean)) @ self.x)

    def predict(self, vector):
        diff = vector - self.ref
        k = self.scale ** 2 * (self.x * self.variance) @ diff
        variance = float(self.scale ** 2 * (diff * self.variance) @ diff - k @ self.inverse @ k)
        approximation_noise = self.tau * np.mean(diff ** 2) / self.step_scale
        return float(self.gradient @ diff), float(np.sqrt(max(variance, 0) + approximation_noise))

if RUN_FORECAST_DIAGNOSTICS:
    scores = {r['key']: r['score'] for r in observations}
    extra = {r['key']: r['extra_score_variance_points'] for r in observations}
    original = {r['key']: arrays[r['key']] for r in observations if r['exact_original_artifact']}
    for mode, values in [('full', arrays), ('exact', original)]:
        for physical in [False, True]:
            model = HeteroForecast(values, scores, RECIPE['forecast_reference'], extra, physical)
            assert model.predict(arrays[RECIPE['forecast_reference']])[0] == 0
            mean, sd = model.predict(pred)
            expected = RECIPE['profiles'][mode + '_' + str(physical)]
            assert abs(mean - expected['mean']) < 1e-6 and abs(sd - expected['sd']) < 1e-6
            print('Rebuilt forecast', mode, physical, 'gain points', mean, 'SD', sd)


## Blend the two public leaders

Attach the scored CSV outputs from **Board Model + RealMLP** and **Which Signals Rank Fraud?** and set their paths below. Confirm their versions match the screenshot scores. Each CSV is aligned to the sample IDs and converted to percentile ranks. The candidate gives the two leaders 47.5% each and V3 5%. The small V3 weight tests whether its ordering adds useful information. Equal public scores can come from different rankings, and a blend can score lower than either parent. These weights are an experimental choice without labeled validation. Submit `submission_v4.csv` to measure the result.

In [ ]:
from scipy.stats import spearmanr

# Attach the two scored notebook outputs and set these to their exact CSV paths.
BOARD_CSV = "/kaggle/input/notebooks/goodpjw2008/s6e11-board-model-realmlp-lb-0-95559/submission.csv"
SIGNALS_CSV = "/kaggle/input/notebooks/areebbb/s6e11-which-signals-rank-fraud-lb-0-95559/submission.csv"
if not BOARD_CSV or not SIGNALS_CSV:
    raise ValueError("Set BOARD_CSV and SIGNALS_CSV to the two 0.95559 submission.csv paths.")

def load_ranks(path):
    frame = pd.read_csv(path)
    if list(frame.columns) != ["id", "is_fraud"]:
        raise ValueError(f"Unexpected columns in {path}")
    if not frame.id.is_unique or len(frame) != len(sample):
        raise ValueError(f"Invalid ID count in {path}")
    scores = frame.set_index("id").is_fraud.reindex(sample.id).to_numpy()
    if not np.isfinite(scores).all():
        raise ValueError(f"Missing IDs or nonfinite scores in {path}")
    return rank(scores)

board = load_ranks(BOARD_CSV)
signals = load_ranks(SIGNALS_CSV)
candidate = rank(0.475 * board + 0.475 * signals + 0.05 * rank(pred))
if any(np.array_equal(candidate, baseline) for baseline in (board, signals, rank(pred))):
    raise ValueError("Candidate duplicates an input ranking. Skip submitting it.")

print("Donor rank correlation:", spearmanr(board, signals).statistic)
output = sample.assign(is_fraud=candidate)
output.to_csv(O / "submission_v4.csv", index=False, lineterminator="\n")
checked = pd.read_csv(O / "submission_v4.csv")
assert checked.id.equals(sample.id) and np.array_equal(rank(checked.is_fraud), candidate)
print("Verified V4 rows:", len(checked))
print("SHA-256:", hashlib.sha256((O / "submission_v4.csv").read_bytes()).hexdigest())


## Discover and register newer candidates

Attach outputs from newer public notebooks. Run the inventory, then record each exact CSV path, the score tied to that output version, and its source URL in the next cell. The V4 inputs are already pinned in notebook metadata. The inventory cannot infer scores from filenames.

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

paths = sorted(I.rglob("submission*.csv"))
display(pd.DataFrame({"attached_csv": [str(path) for path in paths]}))


## Rebuild the two 0.95567 references

The pinned November 23 Round 2 archive supplies the newer rank vectors. Rebuild Malyshev Danil’s V43 and Areeb Ali’s quarter-step continuation, checking each against the expected SHA-256. When public scores tie, anchor on the quarter-step by registration order. These are source-reported scores.

In [ ]:
from io import StringIO
from scipy.special import logit
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

inputs = Path("/kaggle/input")
files = next(p.parent for p in inputs.rglob("sample_submission.csv") if "playground-series-s6e11" in str(p))
sample = pd.read_csv(files / "sample_submission.csv")
test = pd.read_csv(files / "test.csv")
assert test.id.equals(sample.id) and sample.id.is_unique and len(sample) == 299984
with np.load(next(p for p in inputs.rglob("scored_rankings.npz") if "s6e11-nov21-expanded-public-frontier" in str(p)), allow_pickle=False) as archive:
    assert np.array_equal(archive["id"], sample.id)
    step_z = archive["Z"].copy() if "Z" in archive.files else None
with np.load(next(p for p in inputs.rglob("additional_scored_rankings.npz") if "s6e11-nov23-round2-public-frontier" in str(p)), allow_pickle=False) as archive:
    assert np.array_equal(archive["id"], sample.id)
    v42 = archive["own_v42"].copy()
    if step_z is None:
        step_z = archive["Z"].copy()

def ranks(values):
    return (rankdata(values) - .5) / len(values)

def logit_ranks(values):
    assert np.isfinite(values).all()
    return logit(ranks(values))

def csv_text(probabilities):
    return sample.assign(is_fraud=probabilities).to_csv(index=False, lineterminator="\n")

flags = ["is_new_merchant", "is_foreign_transaction", "used_vpn", "ip_country_mismatch",
         "billing_shipping_mismatch", "is_ai_generated_scam_attempt"]
clean = test.cvv_retry_count.eq(0) & test.prior_disputes.eq(0) & ~test[flags].astype(bool).any(axis=1)
risk, velocity = test.merchant_risk_score, test.velocity_score
zones = ((risk < 13.1) | ((velocity <= 4.1) & (risk <= 20)) |
         (clean & (((velocity <= 4.1) & (risk <= 33.8)) | ((velocity <= 6) & (risk <= 25))))).to_numpy()
assert len(step_z) == len(v42) == len(sample)
v42_logit = logit_ranks(v42)
v43 = ranks(ranks(v42_logit - (logit_ranks(step_z) - v42_logit)) - 2 * zones)
encoded = csv_text(v43)
assert hashlib.sha256(encoded.encode()).hexdigest() == "5689d07cc601becb3b2530b87fa5c03241a79d54fb2aba10fe2773d6eaf0464c"
donor_dir = O / "verified_donors"
donor_dir.mkdir(exist_ok=True)
(donor_dir / "submission_v43.csv").write_bytes(encoded.encode())
base = pd.read_csv(StringIO(encoded)).is_fraud.to_numpy()
base_logit = logit_ranks(base)
delta = base_logit - v42_logit
cap = np.quantile(np.abs(delta[~zones]), .99)
experiment = []
for name, weight, direction, expected in [
    ("Quarter-step", .25, delta, "60ec5bd11eca0445c3b0b222a2715d3ffeee9594968cbce56061db16d87f5c62"),
    ("Capped half-step", .50, np.clip(delta, -cap, cap), "0edfc94a483bc135adb658725fb4b9e81511dc8e9d76756c6caa5f1f46c368ba"),
]:
    score = base_logit.copy()
    score[~zones] += weight * direction[~zones]
    score[zones] -= 100
    assert score[zones].max() < score[~zones].min()
    np.testing.assert_array_equal(rankdata(score[zones]), rankdata(base_logit[zones]))
    output = csv_text(ranks(score))
    assert hashlib.sha256(output.encode()).hexdigest() == expected
    check = pd.read_csv(StringIO(output))
    assert check.columns.equals(sample.columns) and check.id.equals(sample.id)
    assert check.id.is_unique and len(check) == 299984 and check.is_fraud.between(0,1).all()
    movement = np.abs(rankdata(score) - rankdata(base))
    experiment.append((name, np.median(movement), np.quantile(movement, .99)))
    if name == "Quarter-step":
        path = donor_dir / "submission_quarter_step.csv"
        path.write_bytes(output.encode())
        assert hashlib.sha256(path.read_bytes()).hexdigest() == expected
        print("VERIFIED QUARTER-STEP", len(check), expected)


## Search around the strongest candidate

The strongest registered public score determines the anchor. The search tests 2.5%, 5%, 10% and 20% mixtures with V4 and other registered candidates. It requires a changed ranking and caps the mean absolute percentile shift at 0.001. Candidates with less than 10% changed row ranks are excluded to avoid near-duplicate edits. Among the remaining candidates it prefers the highest weighted average of component public scores. This number is a heuristic for choosing a candidate. It cannot estimate blend ROC-AUC. The thresholds and grid are fixed experimental choices, and the plot shows the actual rank changes.

In [ ]:
# Fill with verified notebook output versions. Use one entry per scored CSV.
paths.extend(sorted(donor_dir.glob("submission*.csv")))
NEW_CANDIDATES = [
    {"path": str(donor_dir / "submission_quarter_step.csv"), "score": 0.95567,
     "url": "https://www.kaggle.com/code/areebbb/s6e11-which-signals-rank-fraud-lb-0-95567"},
    {"path": str(donor_dir / "submission_v43.csv"), "score": 0.95567,
     "url": "https://www.kaggle.com/code/malyshevdanil/fraud-signal-lab-gpt-2-and-glm-fusion?scriptVersionId=372954959"},
]

if not NEW_CANDIDATES:
    raise ValueError("Attach and register at least one newer scored candidate.")

entries = [{"name": "V4", "score": 0.95559, "ranks": candidate}]
for item in NEW_CANDIDATES:
    score = float(item["score"])
    if not 0 < score <= 1 or not item["url"].startswith("https://www.kaggle.com/"):
        raise ValueError("Supply a valid public score and Kaggle source URL.")
    path = Path(item["path"])
    if path not in paths:
        raise ValueError(f"CSV was not found in the attached input inventory: {path}")
    ranks = load_ranks(path)
    if not any(np.array_equal(ranks, entry["ranks"]) for entry in entries):
        entries.append({"name": path.stem, "score": score, "ranks": ranks})

anchor = max(entries, key=lambda entry: entry["score"])
anchor_ranks = anchor["ranks"]
print("Anchor:", anchor["name"], anchor["score"])
display(pd.DataFrame([
    {"candidate": entry["name"], "source_auc": entry["score"],
     "spearman_with_v4": spearmanr(candidate, entry["ranks"]).statistic,
     "mean_percentile_shift": np.abs(candidate - entry["ranks"]).mean()}
    for entry in entries
]))

trials = []
seen = set()
for entry in entries:
    if entry is anchor:
        continue
    for weight in (0.025, 0.05, 0.10, 0.20):
        blend = rank((1 - weight) * anchor_ranks + weight * entry["ranks"])
        key = hashlib.sha256(blend.tobytes()).hexdigest()
        if key in seen or any(np.array_equal(blend, parent["ranks"]) for parent in entries):
            continue
        seen.add(key)
        shift = np.abs(blend - anchor_ranks).mean()
        changed = np.mean(blend != anchor_ranks)
        source_average = (1 - weight) * anchor["score"] + weight * entry["score"]
        trials.append({"partner": entry["name"], "weight": weight, "shift": shift,
                       "changed": changed, "source_average": source_average})

table = pd.DataFrame(trials)
if table.empty:
    raise ValueError("The search produced no distinct blends. Attach a more diverse candidate.")

eligible = table.query("shift <= 0.001 and changed >= 0.10")
display(table)
ax = table.plot.scatter(x="shift", y="changed", c="weight", colormap="viridis",
                        title="Blend rank changes relative to the strongest input")
ax.set(xlabel="Mean absolute percentile shift", ylabel="Fraction of rows with a changed rank")
plt.show()
if eligible.empty:
    raise ValueError("No blend passed the rank-change checks.")

best = eligible.sort_values(["source_average", "shift"], ascending=[False, True]).iloc[0]
partner = next(entry for entry in entries if entry["name"] == best["partner"])
weight = float(best["weight"])
final_ranks = rank((1 - weight) * anchor_ranks + weight * partner["ranks"])

target = O / "submission.csv"
pending = O / "submission.pending"
sample.assign(is_fraud=final_ranks).to_csv(pending, index=False, lineterminator="\n")
verified = pd.read_csv(pending)
if (list(verified.columns) != ["id", "is_fraud"]
        or not verified.id.equals(sample.id)
        or not verified.is_fraud.between(0, 1).all()
        or not np.array_equal(rank(verified.is_fraud), final_ranks)):
    raise ValueError("V5 CSV failed verification.")
pending.replace(target)
print("Verified V5 rows:", len(verified))
print("Selected partner and weight:", partner["name"], weight)
print("SHA-256:", hashlib.sha256(target.read_bytes()).hexdigest())
display(pd.DataFrame(NEW_CANDIDATES)[["score", "path", "url"]])
